# INITIAL CONFIGURATION

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

# %load_ext autoreload
# %autoreload 2
%matplotlib inline

import plotly.graph_objects as go
from plotly.subplots import make_subplots

/home/turbotowerlnx/Documents/Master/SMA/SMA-MultiAgent-Based-Crowd-Simulation/notebooks
/home/turbotowerlnx/Documents/Master/SMA/SMA-MultiAgent-Based-Crowd-Simulation/app


# ANALYSIS

In [2]:
from maikol_utils.file_utils import list_dir_files
from src.config import Configuration

CONFIG = Configuration()

exps, _ = list_dir_files(CONFIG.LOGS_DIR)
exps = [exp for exp in exps if ".csv" in exp]

experiments = {
    os.path.basename(exp).replace(".csv", ""): exp
    for exp in exps
}
experiments

{'experiments_results_a*_vs_bfs_path_finding': '../logs/experiments_results_a*_vs_bfs_path_finding.csv',
 'experiments_results_agent_densities_corridor': '../logs/experiments_results_agent_densities_corridor.csv',
 'experiments_results_agent_densities_corridor_all': '../logs/experiments_results_agent_densities_corridor_all.csv',
 'experiments_results_agent_densities_mall': '../logs/experiments_results_agent_densities_mall.csv',
 'experiments_results_agent_densities_mall_all': '../logs/experiments_results_agent_densities_mall_all.csv',
 'experiments_results_agent_densities_seats': '../logs/experiments_results_agent_densities_seats.csv',
 'experiments_results_agent_densities_seats_all': '../logs/experiments_results_agent_densities_seats_all.csv',
 'experiments_results_agent_densities_snake': '../logs/experiments_results_agent_densities_snake.csv',
 'experiments_results_agent_densities_snake_all': '../logs/experiments_results_agent_densities_snake_all.csv',
 'experiments_results_aggressiv

### Number of agents

In [15]:
different_agents = False
df_n_agents = pd.concat([
    pd.read_csv(experiments['experiments_results_agent_densities_mall'+('_all' if different_agents else '')]),
    pd.read_csv(experiments['experiments_results_agent_densities_seats'+('_all' if different_agents else '')]),
    pd.read_csv(experiments['experiments_results_agent_densities_snake'+('_all' if different_agents else '')]),
    pd.read_csv(experiments['experiments_results_agent_densities_corridor'+('_all' if different_agents else '')]),
])


df_n_agents.groupby(["RunId", "iteration", "initial_agents","scenario_type"])["Step"].max()
# df_n_agents.head(200)

RunId  iteration  initial_agents  scenario_type
0      0          10              CORRIDOR          30
                                  MALL              16
                                  SEATS             36
                                  SNAKE             81
1      0          25              CORRIDOR          21
                                                  ... 
9998   999        200             SNAKE            149
9999   999        300             CORRIDOR          39
                                  MALL              69
                                  SEATS            278
                                  SNAKE            192
Name: Step, Length: 40000, dtype: int64

In [16]:

# Get max steps per run by scenario and agent count
max_steps_per_run = df_n_agents.groupby(["RunId", "initial_agents", "scenario_type"])["Step"].max().reset_index()
max_steps_per_run.columns = ["RunId", "initial_agents", "scenario_type", "max_steps"]
max_steps_per_run.head(10)


,RunId,initial_agents,scenario_type,max_steps
0,0,10,CORRIDOR,30
1,0,10,MALL,16
2,0,10,SEATS,36
3,0,10,SNAKE,81
4,1,25,CORRIDOR,21
5,1,25,MALL,21
6,1,25,SEATS,56
7,1,25,SNAKE,113
8,2,50,CORRIDOR,24
9,2,50,MALL,31


In [17]:


# Overall statistics per experiment type (initial_agents)
stats_per_experiment = max_steps_per_run.groupby("initial_agents")["max_steps"].agg([
    "count",      # number of runs
    "mean",       # mean max steps
    "median",     # median max steps
    "std",        # standard deviation
    "min",        # minimum max steps
    "max",        # maximum max steps
    ("q25", lambda x: x.quantile(0.25)),  # 25th percentile
    ("q75", lambda x: x.quantile(0.75)),  # 75th percentile
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER INITIAL AGENT COUNT (All Scenarios Combined)")
print("=" * 80)
print(stats_per_experiment)

# Statistics by scenario type AND agent count
print("\n" + "=" * 80)
print("BREAKDOWN BY SCENARIO TYPE AND INITIAL AGENTS")
print("=" * 80)

scenario_types = sorted(df_n_agents['scenario_type'].unique())
for scenario in scenario_types:
    scenario_data = max_steps_per_run[max_steps_per_run['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("initial_agents")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario.upper()}:")
    print(scenario_stats)


STATISTICS OF MAX STEPS PER INITIAL AGENT COUNT (All Scenarios Combined)
                count    mean  median    std  min  max   q25    q75
initial_agents                                                     
10               4000   47.17    35.0  29.72    9  157  23.0   66.0
25               4000   54.01    43.0  32.34   16  150  26.0   78.0
50               4000   59.44    50.0  34.57   19  141  28.0   89.0
75               4000   64.76    55.0  37.22   20  154  30.0  100.0
100              4000   70.60    66.5  40.60   23  155  31.0  110.0
125              4000   77.64    66.0  44.76   24  164  34.0  122.0
150              4000   85.47    73.5  48.80   29  179  37.0  133.0
175              4000   94.14    78.5  53.51   33  196  41.0  147.0
200              4000  102.05    93.5  59.49   33  229  43.0  159.0
300              4000  133.90   121.5  84.22   33  286  55.0  212.0

BREAKDOWN BY SCENARIO TYPE AND INITIAL AGENTS

CORRIDOR:
                count   mean  median   std  min  max


In [18]:
# Create subplots: overall visualization and by-scenario breakdown
num_scenarios = len(scenario_types)
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Distribution of Max Steps by Initial Agents (All Scenarios)",
        "Mean Max Steps by Initial Agents (All Scenarios)",
        "Mean Max Steps by Scenario and Initial Agents",
        "Box Plot by Scenario Type"
    ),
    specs=[[{}, {}], [{}, {}]]
)

# Plot 1: Box plot (overall)
for agent_count in sorted(max_steps_per_run["initial_agents"].unique()):
    data = max_steps_per_run[max_steps_per_run["initial_agents"] == agent_count]["max_steps"]
    fig.add_trace(
        go.Box(y=data, name=f"{agent_count}", showlegend=False),
        row=1, col=1
    )

# Plot 2: Line plot with error bars (overall)
stats_overall = max_steps_per_run.groupby("initial_agents")["max_steps"].agg(["mean", "std"]).reset_index()
fig.add_trace(
    go.Scatter(
        x=stats_overall["initial_agents"],
        y=stats_overall["mean"],
        error_y=dict(type="data", array=stats_overall["std"], visible=True),
        mode="lines+markers",
        name="Mean ± Std",
        marker=dict(size=8),
        line=dict(width=2),
        showlegend=True
    ),
    row=1, col=2
)

# Plot 3: Line plot by scenario type
colors = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan"]
for i, scenario in enumerate(scenario_types):
    scenario_data = max_steps_per_run[max_steps_per_run['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("initial_agents")["max_steps"].agg(["mean", "std"]).reset_index()
    
    fig.add_trace(
        go.Scatter(
            x=scenario_stats["initial_agents"],
            y=scenario_stats["mean"],
            error_y=dict(type="data", array=scenario_stats["std"], visible=True),
            mode="lines+markers",
            name=scenario,
            marker=dict(size=8),
            line=dict(width=2, color=colors[i % len(colors)]),
            showlegend=True
        ),
        row=1, col=2
    )

# Plot 4: Box plot by scenario type
for scenario in scenario_types:
    scenario_data = max_steps_per_run[max_steps_per_run['scenario_type'] == scenario]
    fig.add_trace(
        go.Box(
            y=scenario_data["max_steps"],
            name=scenario,
            showlegend=False
        ),
        row=2, col=2
    )

# Update layout
fig.update_xaxes(title_text="Initial Agents", row=1, col=1)
fig.update_yaxes(title_text="Max Steps", row=1, col=1)
fig.update_xaxes(title_text="Initial Agents", row=1, col=2)
fig.update_yaxes(title_text="Max Steps", row=1, col=2)
fig.update_xaxes(title_text="Initial Agents", row=2, col=1)
fig.update_yaxes(title_text="Max Steps", row=2, col=1)
fig.update_xaxes(title_text="Scenario Type", row=2, col=2)
fig.update_yaxes(title_text="Max Steps", row=2, col=2)

fig.update_layout(height=800, width=1400, showlegend=True, 
                  title_text="Agent Density Impact Analysis - By Scenario Type and Initial Agents")
fig.show()


In [19]:


# Detailed comparison analysis: scenario vs agent count interaction
print("\n" + "=" * 100)
print("INTERACTION ANALYSIS: SCENARIO TYPE vs INITIAL AGENT COUNT")
print("=" * 100)

# Calculate statistics for each combination
interaction_stats = []
for scenario in scenario_types:
    scenario_data = max_steps_per_run[max_steps_per_run['scenario_type'] == scenario]
    agent_counts = sorted(scenario_data['initial_agents'].unique())
    
    print(f"\n{scenario.upper()}:")
    print("-" * 100)
    
    for agent_count in agent_counts:
        combo_data = scenario_data[scenario_data['initial_agents'] == agent_count]['max_steps']
        
        stats_dict = {
            'scenario': scenario,
            'initial_agents': agent_count,
            'count': len(combo_data),
            'mean': combo_data.mean(),
            'std': combo_data.std(),
            'median': combo_data.median(),
            'min': combo_data.min(),
            'max': combo_data.max()
        }
        interaction_stats.append(stats_dict)
        
        print(f"  {agent_count:3.0f} agents: mean={combo_data.mean():7.1f}, std={combo_data.std():7.1f}, "
              f"median={combo_data.median():7.1f}, range=[{combo_data.min():.0f}, {combo_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df = pd.DataFrame(interaction_stats)

# Calculate growth metrics
print("\n" + "=" * 100)
print("GROWTH ANALYSIS: EFFECT OF INCREASING AGENT COUNT")
print("=" * 100)

for scenario in scenario_types:
    scenario_data = interaction_df[interaction_df['scenario'] == scenario]
    
    if len(scenario_data) > 1:
        baseline = scenario_data.iloc[0]['mean']
        max_mean = scenario_data['mean'].max()
        growth_pct = ((max_mean - baseline) / baseline) * 100
        
        print(f"\n{scenario}:")
        print(f"  Baseline ({scenario_data.iloc[0]['initial_agents']:.0f} agents): {baseline:.1f} steps")
        print(f"  Maximum ({scenario_data[scenario_data['mean'] == max_mean].iloc[0]['initial_agents']:.0f} agents): {max_mean:.1f} steps")
        print(f"  Growth: {growth_pct:+.1f}%")

# Scenario comparison at each agent count
print("\n" + "=" * 100)
print("SCENARIO COMPARISON AT EACH AGENT COUNT")
print("=" * 100)

for agent_count in sorted(max_steps_per_run['initial_agents'].unique()):
    agent_data = interaction_df[interaction_df['initial_agents'] == agent_count]
    
    if len(agent_data) > 0:
        best_scenario = agent_data.loc[agent_data['mean'].idxmin()]
        worst_scenario = agent_data.loc[agent_data['mean'].idxmax()]
        diff_pct = ((worst_scenario['mean'] - best_scenario['mean']) / best_scenario['mean']) * 100
        
        print(f"\n{agent_count:.0f} agents:")
        print(f"  Best performer:  {best_scenario['scenario']:20s} - {best_scenario['mean']:.1f} steps (±{best_scenario['std']:.1f})")
        print(f"  Worst performer: {worst_scenario['scenario']:20s} - {worst_scenario['mean']:.1f} steps (±{worst_scenario['std']:.1f})")
        print(f"  Difference: {diff_pct:+.1f}%")



INTERACTION ANALYSIS: SCENARIO TYPE vs INITIAL AGENT COUNT

CORRIDOR:
----------------------------------------------------------------------------------------------------
   10 agents: mean=   22.6, std=    4.9, median=   22.0, range=[9, 51]
   25 agents: mean=   25.9, std=    4.7, median=   25.0, range=[16, 45]
   50 agents: mean=   28.1, std=    4.5, median=   27.0, range=[19, 52]
   75 agents: mean=   29.4, std=    4.2, median=   29.0, range=[20, 47]
  100 agents: mean=   31.0, std=    4.0, median=   30.0, range=[23, 48]
  125 agents: mean=   33.3, std=    3.9, median=   33.0, range=[24, 51]
  150 agents: mean=   37.2, std=    3.7, median=   37.0, range=[29, 53]
  175 agents: mean=   40.7, std=    3.7, median=   40.0, range=[33, 58]
  200 agents: mean=   40.7, std=    3.9, median=   40.0, range=[33, 60]
  300 agents: mean=   40.8, std=    4.0, median=   40.0, range=[33, 57]

MALL:
----------------------------------------------------------------------------------------------------
 

### Aggressive impact

In [20]:
df_aggressive = pd.concat([
    # pd.read_csv(experiments['experiments_results_aggressive_agents_impact_1']),
    # pd.read_csv(experiments['experiments_results_aggressive_agents_impact_2']),
    pd.read_csv(experiments['experiments_results_aggressive_agents_impact']),
])


df_aggressive.groupby(["RunId", "iteration", "initial_agents","scenario_type", 'aggressive_ratio'])["Step"].max()
# df_aggressive.head(200)

RunId  iteration  initial_agents  scenario_type  aggressive_ratio
0      0          125             SEATS          1.00                106
1      0          125             SEATS          0.80                 88
2      0          125             SEATS          0.50                104
3      0          125             SEATS          0.25                100
4      0          125             SEATS          0.10                113
                                                                    ... 
17995  999        300             SEATS          0.80                218
17996  999        300             SEATS          0.50                231
17997  999        300             SEATS          0.25                234
17998  999        300             SEATS          0.10                245
17999  999        300             SEATS          0.00                261
Name: Step, Length: 18000, dtype: int64

In [21]:

# Get max steps per run by scenario, agent count, and aggressive ratio
max_steps_aggressive = df_aggressive.groupby(["RunId", "initial_agents", "scenario_type", "aggressive_ratio"])["Step"].max().reset_index()
max_steps_aggressive.columns = ["RunId", "initial_agents", "scenario_type", "aggressive_ratio", "max_steps"]
max_steps_aggressive.head(15)


,RunId,initial_agents,scenario_type,aggressive_ratio,max_steps
0,0,125,SEATS,1.00,106
1,1,125,SEATS,0.80,88
2,2,125,SEATS,0.50,104
3,3,125,SEATS,0.25,100
4,4,125,SEATS,0.10,113
5,5,125,SEATS,0.00,117
6,6,200,SEATS,1.00,134
7,7,200,SEATS,0.80,144
8,8,200,SEATS,0.50,140
9,9,200,SEATS,0.25,150


In [22]:

# Overall statistics per aggressive ratio
stats_per_aggressive = max_steps_aggressive.groupby("aggressive_ratio")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER AGGRESSIVE RATIO (All Scenarios Combined)")
print("=" * 80)
print(stats_per_aggressive)

# Statistics by scenario type AND aggressive ratio
print("\n" + "=" * 80)
print("BREAKDOWN BY SCENARIO TYPE AND AGGRESSIVE RATIO")
print("=" * 80)

scenario_types_agg = sorted(df_aggressive['scenario_type'].unique())
for scenario in scenario_types_agg:
    scenario_data = max_steps_aggressive[max_steps_aggressive['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("aggressive_ratio")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario.upper()}:")
    print(scenario_stats)


STATISTICS OF MAX STEPS PER AGGRESSIVE RATIO (All Scenarios Combined)
                  count    mean  median    std  min  max     q25    q75
aggressive_ratio                                                       
0.00               3000  170.89   163.0  54.25   83  278  118.00  228.0
0.10               3000  165.91   158.0  52.76   80  278  114.00  221.0
0.25               3000  160.19   152.0  51.16   76  262  109.75  214.0
0.50               3000  153.92   146.0  48.97   73  260  106.00  205.0
0.80               3000  149.29   142.0  48.34   71  252  102.00  200.0
1.00               3000  146.61   139.0  47.07   69  251  100.00  196.0

BREAKDOWN BY SCENARIO TYPE AND AGGRESSIVE RATIO

SEATS:
                  count    mean  median    std  min  max
aggressive_ratio                                        
0.00               3000  170.89   163.0  54.25   83  278
0.10               3000  165.91   158.0  52.76   80  278
0.25               3000  160.19   152.0  51.16   76  262
0.50        

In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots: one per scenario type with aggressive ratio comparison
num_scenarios_agg = len(scenario_types_agg)

# Calculate dimensions for subplots (2 columns, rows as needed)
num_cols = 2
num_rows = (num_scenarios_agg + num_cols - 1) // num_cols

from plotly.subplots import make_subplots

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[s.upper() for s in scenario_types_agg],
    specs=[[{} for _ in range(num_cols)] for _ in range(num_rows)]
)

# Get unique aggressive ratios
aggressive_ratios = sorted(max_steps_aggressive['aggressive_ratio'].unique())
colors = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan", "magenta", "yellow"]

# Create a plot for each scenario
for idx, scenario in enumerate(scenario_types_agg):
    row = (idx // num_cols) + 1
    col = (idx % num_cols) + 1
    
    scenario_data = max_steps_aggressive[max_steps_aggressive['scenario_type'] == scenario]
    
    # Add a line for each aggressive ratio
    for ratio_idx, ratio in enumerate(aggressive_ratios):
        ratio_data = scenario_data[scenario_data['aggressive_ratio'] == ratio]
        ratio_stats = ratio_data.groupby("initial_agents")["max_steps"].agg(["mean", "std"]).reset_index()
        
        fig.add_trace(
            go.Scatter(
                x=ratio_stats["initial_agents"],
                y=ratio_stats["mean"],
                error_y=dict(type="data", array=ratio_stats["std"], visible=True),
                mode="lines+markers",
                name=f"Ratio: {ratio}",
                marker=dict(size=6),
                line=dict(width=2, color=colors[ratio_idx % len(colors)]),
                showlegend=(idx == 0),  # Show legend only for first subplot
                legendgroup=f"ratio_{ratio}",
            ),
            row=row, col=col
        )
    
    fig.update_xaxes(title_text="Initial Agents", row=row, col=col)
    fig.update_yaxes(title_text="Max Steps", row=row, col=col)

fig.update_layout(height=300*num_rows*3, width=1400, showlegend=True,
                  title_text="Aggressive Agents Impact Analysis - Max Steps by Agent Count and Aggressive Ratio")
fig.show()


In [24]:

# Detailed comparison analysis: scenario vs agent count vs aggressive ratio
print("\n" + "=" * 120)
print("INTERACTION ANALYSIS: SCENARIO TYPE vs INITIAL AGENT COUNT vs AGGRESSIVE RATIO")
print("=" * 120)

# Calculate statistics for each combination
interaction_stats_agg = []
for scenario in scenario_types_agg:
    scenario_data = max_steps_aggressive[max_steps_aggressive['scenario_type'] == scenario]
    agent_counts = sorted(scenario_data['initial_agents'].unique())
    
    print(f"\n{scenario.upper()}:")
    print("-" * 120)
    
    for agent_count in agent_counts:
        agent_data = scenario_data[scenario_data['initial_agents'] == agent_count]
        
        print(f"\n  {agent_count:.0f} agents:")
        
        for ratio in aggressive_ratios:
            combo_data = agent_data[agent_data['aggressive_ratio'] == ratio]['max_steps']
            
            if len(combo_data) > 0:
                stats_dict = {
                    'scenario': scenario,
                    'initial_agents': agent_count,
                    'aggressive_ratio': ratio,
                    'count': len(combo_data),
                    'mean': combo_data.mean(),
                    'std': combo_data.std(),
                    'median': combo_data.median(),
                    'min': combo_data.min(),
                    'max': combo_data.max()
                }
                interaction_stats_agg.append(stats_dict)
                
                print(f"    Ratio {ratio:4.2f}: mean={combo_data.mean():7.1f}, std={combo_data.std():7.1f}, "
                      f"median={combo_data.median():7.1f}, range=[{combo_data.min():.0f}, {combo_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df_agg = pd.DataFrame(interaction_stats_agg)

# Analyze effect of increasing aggressive ratio at each agent count
print("\n" + "=" * 120)
print("IMPACT ANALYSIS: EFFECT OF INCREASING AGGRESSIVE RATIO")
print("=" * 120)

for scenario in scenario_types_agg:
    print(f"\n{scenario.upper()}:")
    
    scenario_data = interaction_df_agg[interaction_df_agg['scenario'] == scenario]
    agent_counts_unique = sorted(scenario_data['initial_agents'].unique())
    
    for agent_count in agent_counts_unique:
        agent_data = scenario_data[scenario_data['initial_agents'] == agent_count]
        
        if len(agent_data) > 1:
            baseline = agent_data[agent_data['aggressive_ratio'] == agent_data['aggressive_ratio'].min()]['mean'].values[0]
            max_mean = agent_data['mean'].max()
            max_ratio = agent_data[agent_data['mean'] == max_mean]['aggressive_ratio'].values[0]
            growth_pct = ((max_mean - baseline) / baseline) * 100
            
            print(f"  {agent_count:.0f} agents:")
            print(f"    Baseline (ratio {agent_data['aggressive_ratio'].min():.2f}): {baseline:.1f} steps")
            print(f"    Maximum (ratio {max_ratio:.2f}): {max_mean:.1f} steps")
            print(f"    Impact: {growth_pct:+.1f}%")



INTERACTION ANALYSIS: SCENARIO TYPE vs INITIAL AGENT COUNT vs AGGRESSIVE RATIO

SEATS:
------------------------------------------------------------------------------------------------------------------------

  125 agents:
    Ratio 0.00: mean=  110.2, std=   11.2, median=  109.0, range=[83, 154]
    Ratio 0.10: mean=  107.2, std=   10.7, median=  106.0, range=[80, 157]
    Ratio 0.25: mean=  103.4, std=   10.4, median=  103.0, range=[76, 155]
    Ratio 0.50: mean=   99.2, std=   10.3, median=   99.0, range=[73, 134]
    Ratio 0.80: mean=   95.5, std=    9.7, median=   95.0, range=[71, 133]
    Ratio 1.00: mean=   94.2, std=    9.7, median=   94.0, range=[69, 140]

  200 agents:
    Ratio 0.00: mean=  164.2, std=   14.1, median=  163.0, range=[128, 223]
    Ratio 0.10: mean=  159.1, std=   14.3, median=  158.0, range=[121, 214]
    Ratio 0.25: mean=  153.1, std=   13.6, median=  152.0, range=[117, 202]
    Ratio 0.50: mean=  148.0, std=   12.8, median=  146.0, range=[116, 203]
    Rat

### Slow impact

In [25]:

df_slow = pd.concat([
    pd.read_csv(experiments['experiments_results_slow_agents_impact_1']),
    pd.read_csv(experiments['experiments_results_slow_agents_impact_2']),
])

df_slow.groupby(["RunId", "iteration", "initial_agents","scenario_type", 'slow_ratio'])["Step"].max()
# df_slow.head(200)


RunId  iteration  initial_agents  scenario_type  slow_ratio
0      0          125             SEATS          0.00          117
                                                 0.50          156
1      0          125             SEATS          0.10          127
                                                 0.80          172
2      0          125             SEATS          0.25          126
                                                              ... 
8997   999        300             SEATS          0.50          353
8998   999        300             SEATS          0.10          280
                                                 0.80          379
8999   999        300             SEATS          0.25          351
                                                 1.00          403
Name: Step, Length: 18000, dtype: int64

In [26]:

# Get max steps per run by scenario, agent count, and slow ratio
max_steps_slow = df_slow.groupby(["RunId", "initial_agents", "scenario_type", "slow_ratio"])["Step"].max().reset_index()
max_steps_slow.columns = ["RunId", "initial_agents", "scenario_type", "slow_ratio", "max_steps"]
max_steps_slow.head(15)


,RunId,initial_agents,scenario_type,slow_ratio,max_steps
0,0,125,SEATS,0.00,117
1,0,125,SEATS,0.50,156
2,1,125,SEATS,0.10,127
3,1,125,SEATS,0.80,172
4,2,125,SEATS,0.25,126
5,2,125,SEATS,1.00,191
6,3,200,SEATS,0.00,148
7,3,200,SEATS,0.50,211
8,4,200,SEATS,0.10,176
9,4,200,SEATS,0.80,227


In [27]:

# Overall statistics per slow ratio
stats_per_slow = max_steps_slow.groupby("slow_ratio")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER SLOW RATIO (All Scenarios Combined)")
print("=" * 80)
print(stats_per_slow)

# Statistics by scenario type AND slow ratio
print("\n" + "=" * 80)
print("BREAKDOWN BY SCENARIO TYPE AND SLOW RATIO")
print("=" * 80)

scenario_types_slow = sorted(df_slow['scenario_type'].unique())
for scenario in scenario_types_slow:
    scenario_data = max_steps_slow[max_steps_slow['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("slow_ratio")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario.upper()}:")
    print(scenario_stats)


STATISTICS OF MAX STEPS PER SLOW RATIO (All Scenarios Combined)
            count    mean  median    std  min  max    q25    q75
slow_ratio                                                      
0.00         3000  170.58   163.0  54.21   79  298  117.0  228.0
0.10         3000  191.10   182.0  57.98   94  337  137.0  250.0
0.25         3000  209.41   199.0  62.46   99  360  151.0  271.0
0.50         3000  228.27   217.0  68.42  104  398  164.0  296.0
0.80         3000  242.63   229.0  72.41  125  417  175.0  315.0
1.00         3000  250.45   238.0  75.81  123  431  178.0  325.0

BREAKDOWN BY SCENARIO TYPE AND SLOW RATIO

SEATS:
            count    mean  median    std  min  max
slow_ratio                                        
0.00         3000  170.58   163.0  54.21   79  298
0.10         3000  191.10   182.0  57.98   94  337
0.25         3000  209.41   199.0  62.46   99  360
0.50         3000  228.27   217.0  68.42  104  398
0.80         3000  242.63   229.0  72.41  125  417
1.00    

In [28]:

# Create subplots: one per scenario type with slow ratio comparison
num_scenarios_slow = len(scenario_types_slow)

# Calculate dimensions for subplots (2 columns, rows as needed)        # {
        #     "title": "Aggressive Agents Impact 2",
        #     "batch_params": {
        #         "width": CONFIG.width,
        #         "height": CONFIG.height,
        #         "initial_agents": [125, 200, 300],
        #         "track_last_steps": CONFIG.track_last_steps,
        #         "path_finding_algorithm": CONFIG.path_finding_algorithm,
        #         "differentiate_exits": CONFIG.differentiate_exits,
        #         "respawn_agents": CONFIG.respawn_agents,
        #         "polite_ratio": 1.0,
        #         "aggressive_ratio": [0.5, 0.8, 1.0],
        #         "slow_ratio": 0.0,
        #         "scenario_type": ["SEATS"],
        #         "n_exits": CONFIG.n_exits,
        #     }
        # },
        # {
        #     "title": "Slow Agents Impact 1",
        #     "batch_params": {
        #         "width": CONFIG.width,
        #         "height": CONFIG.height,
        #         "initial_agents": [125, 200, 300],
        #         "track_last_steps": CONFIG.track_last_steps,
        #         "path_finding_algorithm": CONFIG.path_finding_algorithm,
        #         "differentiate_exits": CONFIG.differentiate_exits,
        #         "respawn_agents": CONFIG.respawn_agents,
        #         "polite_ratio": 1.0,
        #         "aggressive_ratio": 0.0,
        #         "slow_ratio": [0.0, 0.1, 0.25],
        #         "scenario_type": ["SEATS"],
        #         "n_exits": CONFIG.n_exits,
        #     }
        # },
        # {
        #     "title": "Slow Agents Impact 2",
        #     "batch_params": {
        #         "width": CONFIG.width,
        #         "height": CONFIG.height,
        #         "initial_agents": [125, 200, 300],
        #         "track_last_steps": CONFIG.track_last_steps,
        #         "path_finding_algorithm": CONFIG.path_finding_algorithm,
        #         "differentiate_exits": CONFIG.differentiate_exits,
        #         "respawn_agents": CONFIG.respawn_agents,
        #         "polite_ratio": 1.0,
        #         "aggressive_ratio": 0.0,
        #         "slow_ratio": [0.5, 0.8, 1.0],
        #         "scenario_type": ["SEATS"],
        #         "n_exits": CONFIG.n_exits,
        #     }
        # },
num_cols = 2
num_rows = (num_scenarios_slow + num_cols - 1) // num_cols

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[s.upper() for s in scenario_types_slow],
    specs=[[{} for _ in range(num_cols)] for _ in range(num_rows)]
)

# Get unique slow ratios
slow_ratios = sorted(max_steps_slow['slow_ratio'].unique())
colors = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan", "magenta", "yellow"]

# Create a plot for each scenario
for idx, scenario in enumerate(scenario_types_slow):
    row = (idx // num_cols) + 1
    col = (idx % num_cols) + 1
    
    scenario_data = max_steps_slow[max_steps_slow['scenario_type'] == scenario]
    
    # Add a line for each slow ratio
    for ratio_idx, ratio in enumerate(slow_ratios):
        ratio_data = scenario_data[scenario_data['slow_ratio'] == ratio]
        ratio_stats = ratio_data.groupby("initial_agents")["max_steps"].agg(["mean", "std"]).reset_index()
        
        fig.add_trace(
            go.Scatter(
                x=ratio_stats["initial_agents"],
                y=ratio_stats["mean"],
                error_y=dict(type="data", array=ratio_stats["std"], visible=True),
                mode="lines+markers",
                name=f"Ratio: {ratio}",
                marker=dict(size=6),
                line=dict(width=2, color=colors[ratio_idx % len(colors)]),
                showlegend=(idx == 0),  # Show legend only for first subplot
                legendgroup=f"ratio_{ratio}",
            ),
            row=row, col=col
        )
    
    fig.update_xaxes(title_text="Initial Agents", row=row, col=col)
    fig.update_yaxes(title_text="Max Steps", row=row, col=col)

fig.update_layout(height=300*num_rows*3, width=1400, showlegend=True,
                  title_text="Slow Agents Impact Analysis - Max Steps by Agent Count and Slow Ratio")
fig.show()


In [29]:

# Detailed comparison analysis: scenario vs agent count vs slow ratio
print("\n" + "=" * 120)
print("INTERACTION ANALYSIS: SCENARIO TYPE vs INITIAL AGENT COUNT vs SLOW RATIO")
print("=" * 120)

# Calculate statistics for each combination
interaction_stats_slow = []
for scenario in scenario_types_slow:
    scenario_data = max_steps_slow[max_steps_slow['scenario_type'] == scenario]
    agent_counts = sorted(scenario_data['initial_agents'].unique())
    
    print(f"\n{scenario.upper()}:")
    print("-" * 120)
    
    for agent_count in agent_counts:
        agent_data = scenario_data[scenario_data['initial_agents'] == agent_count]
        
        print(f"\n  {agent_count:.0f} agents:")
        
        for ratio in slow_ratios:
            combo_data = agent_data[agent_data['slow_ratio'] == ratio]['max_steps']
            
            if len(combo_data) > 0:
                stats_dict = {
                    'scenario': scenario,
                    'initial_agents': agent_count,
                    'slow_ratio': ratio,
                    'count': len(combo_data),
                    'mean': combo_data.mean(),
                    'std': combo_data.std(),
                    'median': combo_data.median(),
                    'min': combo_data.min(),
                    'max': combo_data.max()
                }
                interaction_stats_slow.append(stats_dict)
                
                print(f"    Ratio {ratio:4.2f}: mean={combo_data.mean():7.1f}, std={combo_data.std():7.1f}, "
                      f"median={combo_data.median():7.1f}, range=[{combo_data.min():.0f}, {combo_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df_slow = pd.DataFrame(interaction_stats_slow)

# Analyze effect of increasing slow ratio at each agent count
print("\n" + "=" * 120)
print("IMPACT ANALYSIS: EFFECT OF INCREASING SLOW RATIO")
print("=" * 120)

for scenario in scenario_types_slow:
    print(f"\n{scenario.upper()}:")
    
    scenario_data = interaction_df_slow[interaction_df_slow['scenario'] == scenario]
    agent_counts_unique = sorted(scenario_data['initial_agents'].unique())
    
    for agent_count in agent_counts_unique:
        agent_data = scenario_data[scenario_data['initial_agents'] == agent_count]
        
        if len(agent_data) > 1:
            baseline = agent_data[agent_data['slow_ratio'] == agent_data['slow_ratio'].min()]['mean'].values[0]
            max_mean = agent_data['mean'].max()
            max_ratio = agent_data[agent_data['mean'] == max_mean]['slow_ratio'].values[0]
            growth_pct = ((max_mean - baseline) / baseline) * 100
            
            print(f"  {agent_count:.0f} agents:")
            print(f"    Baseline (ratio {agent_data['slow_ratio'].min():.2f}): {baseline:.1f} steps")
            print(f"    Maximum (ratio {max_ratio:.2f}): {max_mean:.1f} steps")
            print(f"    Impact: {growth_pct:+.1f}%")



INTERACTION ANALYSIS: SCENARIO TYPE vs INITIAL AGENT COUNT vs SLOW RATIO

SEATS:
------------------------------------------------------------------------------------------------------------------------

  125 agents:
    Ratio 0.00: mean=  110.0, std=   10.8, median=  109.5, range=[79, 150]
    Ratio 0.10: mean=  127.8, std=   15.0, median=  126.0, range=[94, 191]
    Ratio 0.25: mean=  141.4, std=   15.7, median=  140.0, range=[99, 198]
    Ratio 0.50: mean=  154.0, std=   16.3, median=  153.0, range=[104, 210]
    Ratio 0.80: mean=  164.5, std=   17.4, median=  163.0, range=[125, 226]
    Ratio 1.00: mean=  167.7, std=   17.6, median=  167.0, range=[123, 252]

  200 agents:
    Ratio 0.00: mean=  163.8, std=   14.2, median=  163.0, range=[124, 215]
    Ratio 0.10: mean=  183.3, std=   18.7, median=  182.0, range=[137, 257]
    Ratio 0.25: mean=  200.6, std=   19.7, median=  199.0, range=[152, 278]
    Ratio 0.50: mean=  217.9, std=   20.4, median=  217.0, range=[164, 296]
    Ratio 

### A* vs BFS

In [30]:
df_a_bfs = pd.concat([
    pd.read_csv(experiments['experiments_results_a*_vs_bfs_path_finding'])
])

df_a_bfs.groupby(["RunId", "iteration","scenario_type", "path_finding_algorithm"])["Step"].max()
# df_a_bfs.head(200)


RunId  iteration  scenario_type  path_finding_algorithm
0      0          MALL           A*                         73
1      0          SEATS          A*                        117
2      0          SNAKE          A*                        193
3      0          CORRIDOR       A*                         49
4      0          MALL           BFS                        66
                                                          ... 
7995   999        CORRIDOR       A*                         43
7996   999        MALL           BFS                        54
7997   999        SEATS          BFS                       116
7998   999        SNAKE          BFS                       198
7999   999        CORRIDOR       BFS                        36
Name: Step, Length: 8000, dtype: int64

In [31]:

# Get max steps per run by scenario and path finding algorithm
max_steps_pathfinding = df_a_bfs.groupby(["RunId", "scenario_type", "path_finding_algorithm"])["Step"].max().reset_index()
max_steps_pathfinding.columns = ["RunId", "scenario_type", "path_finding_algorithm", "max_steps"]
max_steps_pathfinding.head(15)


,RunId,scenario_type,path_finding_algorithm,max_steps
0,0,MALL,A*,73
1,1,SEATS,A*,117
2,2,SNAKE,A*,193
3,3,CORRIDOR,A*,49
4,4,MALL,BFS,66
5,5,SEATS,BFS,115
6,6,SNAKE,BFS,247
7,7,CORRIDOR,BFS,58
8,8,MALL,A*,53
9,9,SEATS,A*,116


In [32]:

# Overall statistics per path finding algorithm
stats_per_pathfinding = max_steps_pathfinding.groupby("path_finding_algorithm")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER PATH FINDING ALGORITHM (All Scenarios Combined)")
print("=" * 80)
print(stats_per_pathfinding)

# Statistics by scenario type AND path finding algorithm
print("\n" + "=" * 80)
print("BREAKDOWN BY SCENARIO TYPE AND PATH FINDING ALGORITHM")
print("=" * 80)

scenario_types_pathfinding = sorted(df_a_bfs['scenario_type'].unique())
for scenario in scenario_types_pathfinding:
    scenario_data = max_steps_pathfinding[max_steps_pathfinding['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("path_finding_algorithm")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario.upper()}:")
    print(scenario_stats)


STATISTICS OF MAX STEPS PER PATH FINDING ALGORITHM (All Scenarios Combined)
                        count    mean  median    std  min  max   q25    q75
path_finding_algorithm                                                     
A*                       4000  101.49    90.5  60.42   27  270  46.0  150.0
BFS                      4000  105.78    88.0  61.25   27  296  56.0  147.0

BREAKDOWN BY SCENARIO TYPE AND PATH FINDING ALGORITHM

CORRIDOR:
                        count   mean  median    std  min  max
path_finding_algorithm                                       
A*                       1000  46.33    45.0  10.64   27   93
BFS                      1000  46.96    45.0  10.06   27   99

MALL:
                        count   mean  median   std  min  max
path_finding_algorithm                                      
A*                       1000  48.61    47.0  9.65   30   87
BFS                      1000  64.96    64.0  9.86   44  108

SEATS:
                        count    mean  median  

In [33]:

# Create subplots: one per scenario type with path finding algorithm comparison
num_scenarios_pathfinding = len(scenario_types_pathfinding)

# Calculate dimensions for subplots (2 columns, rows as needed)
num_cols = 2
num_rows = (num_scenarios_pathfinding + num_cols - 1) // num_cols

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[s.upper() for s in scenario_types_pathfinding],
    specs=[[{} for _ in range(num_cols)] for _ in range(num_rows)]
)

# Get unique algorithms
algorithms = sorted(max_steps_pathfinding['path_finding_algorithm'].unique())
colors_algo = ["blue", "red", "green", "orange"]

# Create a bar plot for each scenario
for idx, scenario in enumerate(scenario_types_pathfinding):
    row = (idx // num_cols) + 1
    col = (idx % num_cols) + 1
    
    scenario_data = max_steps_pathfinding[max_steps_pathfinding['scenario_type'] == scenario]
    
    # Add bars for each algorithm
    for algo_idx, algo in enumerate(algorithms):
        algo_data = scenario_data[scenario_data['path_finding_algorithm'] == algo]
        algo_stats = algo_data.groupby("path_finding_algorithm")["max_steps"].agg(["mean", "std"]).reset_index()
        
        if len(algo_stats) > 0:
            fig.add_trace(
                go.Bar(
                    x=[algo],
                    y=algo_stats["mean"],
                    error_y=dict(type="data", array=algo_stats["std"], visible=True),
                    name=algo,
                    marker=dict(color=colors_algo[algo_idx % len(colors_algo)]),
                    showlegend=(idx == 0),  # Show legend only for first subplot
                ),
                row=row, col=col
            )
    
    fig.update_xaxes(title_text="Path Finding Algorithm", row=row, col=col)
    fig.update_yaxes(title_text="Max Steps", row=row, col=col)

fig.update_layout(height=300*num_rows*3, width=1400, showlegend=True,
                  title_text="Path Finding Algorithm Impact Analysis - Max Steps by Scenario")
fig.show()


In [34]:

# Detailed comparison analysis: scenario vs path finding algorithm
print("\n" + "=" * 120)
print("INTERACTION ANALYSIS: SCENARIO TYPE vs PATH FINDING ALGORITHM")
print("=" * 120)

# Calculate statistics for each combination
interaction_stats_pathfinding = []
for scenario in scenario_types_pathfinding:
    scenario_data = max_steps_pathfinding[max_steps_pathfinding['scenario_type'] == scenario]
    
    print(f"\n{scenario.upper()}:")
    print("-" * 120)
    
    for algo in algorithms:
        algo_data = scenario_data[scenario_data['path_finding_algorithm'] == algo]['max_steps']
        
        if len(algo_data) > 0:
            stats_dict = {
                'scenario': scenario,
                'path_finding_algorithm': algo,
                'count': len(algo_data),
                'mean': algo_data.mean(),
                'std': algo_data.std(),
                'median': algo_data.median(),
                'min': algo_data.min(),
                'max': algo_data.max()
            }
            interaction_stats_pathfinding.append(stats_dict)
            
            print(f"  {algo:20s}: mean={algo_data.mean():7.1f}, std={algo_data.std():7.1f}, "
                  f"median={algo_data.median():7.1f}, range=[{algo_data.min():.0f}, {algo_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df_pathfinding = pd.DataFrame(interaction_stats_pathfinding)

# Algorithm comparison at each scenario
print("\n" + "=" * 120)
print("ALGORITHM COMPARISON BY SCENARIO")
print("=" * 120)

for scenario in scenario_types_pathfinding:
    scenario_data = interaction_df_pathfinding[interaction_df_pathfinding['scenario'] == scenario]
    
    if len(scenario_data) > 1:
        best_algo = scenario_data.loc[scenario_data['mean'].idxmin()]
        worst_algo = scenario_data.loc[scenario_data['mean'].idxmax()]
        diff_pct = ((worst_algo['mean'] - best_algo['mean']) / best_algo['mean']) * 100
        
        print(f"\n{scenario}:")
        print(f"  Best performer:  {best_algo['path_finding_algorithm']:20s} - {best_algo['mean']:.1f} steps (±{best_algo['std']:.1f})")
        print(f"  Worst performer: {worst_algo['path_finding_algorithm']:20s} - {worst_algo['mean']:.1f} steps (±{worst_algo['std']:.1f})")
        print(f"  Difference: {diff_pct:+.1f}%")



INTERACTION ANALYSIS: SCENARIO TYPE vs PATH FINDING ALGORITHM

CORRIDOR:
------------------------------------------------------------------------------------------------------------------------
  A*                  : mean=   46.3, std=   10.6, median=   45.0, range=[27, 93]
  BFS                 : mean=   47.0, std=   10.1, median=   45.0, range=[27, 99]

MALL:
------------------------------------------------------------------------------------------------------------------------
  A*                  : mean=   48.6, std=    9.6, median=   47.0, range=[30, 87]
  BFS                 : mean=   65.0, std=    9.9, median=   64.0, range=[44, 108]

SEATS:
------------------------------------------------------------------------------------------------------------------------
  A*                  : mean=  124.9, std=   15.2, median=  123.0, range=[87, 180]
  BFS                 : mean=  112.0, std=   14.5, median=  111.0, range=[76, 187]

SNAKE:
---------------------------------------------

### Exit preference Impact

In [35]:
df_exits = pd.concat([
    pd.read_csv(experiments['experiments_results_exit_preference_impact'])
])

df_exits.groupby(["RunId", "iteration","scenario_type", "differentiate_exits"])["Step"].max()

RunId  iteration  scenario_type  differentiate_exits
0      0          MALL           True                   104
1      0          SEATS          True                   154
2      0          SNAKE          True                   366
3      0          CORRIDOR       True                   579
4      0          MALL           False                   62
                                                       ... 
7995   999        CORRIDOR       True                   590
7996   999        MALL           False                   65
7997   999        SEATS          False                  119
7998   999        SNAKE          False                  220
7999   999        CORRIDOR       False                   34
Name: Step, Length: 8000, dtype: int64

In [36]:

# Get max steps per run by scenario and exit preference differentiation
max_steps_exits = df_exits.groupby(["RunId", "scenario_type", "differentiate_exits"])["Step"].max().reset_index()
max_steps_exits.columns = ["RunId", "scenario_type", "differentiate_exits", "max_steps"]
max_steps_exits.head(15)


,RunId,scenario_type,differentiate_exits,max_steps
0,0,MALL,True,104
1,1,SEATS,True,154
2,2,SNAKE,True,366
3,3,CORRIDOR,True,579
4,4,MALL,False,62
5,5,SEATS,False,149
6,6,SNAKE,False,216
7,7,CORRIDOR,False,58
8,8,MALL,True,107
9,9,SEATS,True,110


In [37]:

# Overall statistics per exit preference setting
stats_per_exits = max_steps_exits.groupby("differentiate_exits")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER EXIT PREFERENCE SETTING (All Scenarios Combined)")
print("=" * 80)
print(stats_per_exits)

# Statistics by scenario type AND exit preference
print("\n" + "=" * 80)
print("BREAKDOWN BY SCENARIO TYPE AND EXIT PREFERENCE")
print("=" * 80)

scenario_types_exits = sorted(df_exits['scenario_type'].unique())
for scenario in scenario_types_exits:
    scenario_data = max_steps_exits[max_steps_exits['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("differentiate_exits")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario.upper()}:")
    print(scenario_stats)


STATISTICS OF MAX STEPS PER EXIT PREFERENCE SETTING (All Scenarios Combined)
                     count    mean  median     std  min   max    q25    q75
differentiate_exits                                                        
False                 4000  101.62    91.0   60.73   27   270   46.0  149.0
True                  4000  289.29   232.0  184.87   74  1486  126.0  419.0

BREAKDOWN BY SCENARIO TYPE AND EXIT PREFERENCE

CORRIDOR:
                     count    mean  median     std  min   max
differentiate_exits                                          
False                 1000   45.91    45.0   10.27   27    81
True                  1000  520.82   516.0  121.38  199  1486

MALL:
                     count    mean  median    std  min  max
differentiate_exits                                        
False                 1000   48.87    47.0  10.01   29   96
True                  1000  124.71   123.0  25.79   74  276

SEATS:
                     count    mean  median    std  min  m

In [38]:

# Create subplots: one per scenario type with exit preference comparison
num_scenarios_exits = len(scenario_types_exits)

# Calculate dimensions for subplots (2 columns, rows as needed)
num_cols = 2
num_rows = (num_scenarios_exits + num_cols - 1) // num_cols

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[s.upper() for s in scenario_types_exits],
    specs=[[{} for _ in range(num_cols)] for _ in range(num_rows)]
)

# Get unique exit preference settings
exit_settings = sorted(max_steps_exits['differentiate_exits'].unique())
colors_exits = ["blue", "red", "green", "orange"]

# Create a bar plot for each scenario
for idx, scenario in enumerate(scenario_types_exits):
    row = (idx // num_cols) + 1
    col = (idx % num_cols) + 1
    
    scenario_data = max_steps_exits[max_steps_exits['scenario_type'] == scenario]
    
    # Add bars for each exit preference setting
    for setting_idx, setting in enumerate(exit_settings):
        setting_data = scenario_data[scenario_data['differentiate_exits'] == setting]
        setting_stats = setting_data.groupby("differentiate_exits")["max_steps"].agg(["mean", "std"]).reset_index()
        
        if len(setting_stats) > 0:
            setting_label = "With Exit Preference" if setting else "No Exit Preference"
            fig.add_trace(
                go.Bar(
                    x=[setting_label],
                    y=setting_stats["mean"],
                    error_y=dict(type="data", array=setting_stats["std"], visible=True),
                    name=setting_label,
                    marker=dict(color=colors_exits[setting_idx % len(colors_exits)]),
                    showlegend=(idx == 0),  # Show legend only for first subplot
                ),
                row=row, col=col
            )
    
    fig.update_xaxes(title_text="Exit Preference Setting", row=row, col=col)
    fig.update_yaxes(title_text="Max Steps", row=row, col=col)

fig.update_layout(height=300*num_rows*3, width=1400, showlegend=True,
                  title_text="Exit Preference Impact Analysis - Max Steps by Scenario")
fig.show()


In [39]:

# Detailed comparison analysis: scenario vs exit preference
print("\n" + "=" * 120)
print("INTERACTION ANALYSIS: SCENARIO TYPE vs EXIT PREFERENCE")
print("=" * 120)

# Calculate statistics for each combination
interaction_stats_exits = []
for scenario in scenario_types_exits:
    scenario_data = max_steps_exits[max_steps_exits['scenario_type'] == scenario]
    
    print(f"\n{scenario.upper()}:")
    print("-" * 120)
    
    for setting in exit_settings:
        setting_data = scenario_data[scenario_data['differentiate_exits'] == setting]['max_steps']
        
        if len(setting_data) > 0:
            setting_label = "With Exit Preference" if setting else "No Exit Preference"
            stats_dict = {
                'scenario': scenario,
                'exit_preference': setting_label,
                'differentiate_exits': setting,
                'count': len(setting_data),
                'mean': setting_data.mean(),
                'std': setting_data.std(),
                'median': setting_data.median(),
                'min': setting_data.min(),
                'max': setting_data.max()
            }
            interaction_stats_exits.append(stats_dict)
            
            print(f"  {setting_label:25s}: mean={setting_data.mean():7.1f}, std={setting_data.std():7.1f}, "
                  f"median={setting_data.median():7.1f}, range=[{setting_data.min():.0f}, {setting_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df_exits = pd.DataFrame(interaction_stats_exits)

# Exit preference comparison at each scenario
print("\n" + "=" * 120)
print("EXIT PREFERENCE IMPACT BY SCENARIO")
print("=" * 120)

for scenario in scenario_types_exits:
    scenario_data = interaction_df_exits[interaction_df_exits['scenario'] == scenario]
    
    if len(scenario_data) > 1:
        with_pref = scenario_data[scenario_data['differentiate_exits'] == True]
        without_pref = scenario_data[scenario_data['differentiate_exits'] == False]
        
        if len(with_pref) > 0 and len(without_pref) > 0:
            with_pref_mean = with_pref.iloc[0]['mean']
            without_pref_mean = without_pref.iloc[0]['mean']
            diff_pct = ((with_pref_mean - without_pref_mean) / without_pref_mean) * 100
            
            print(f"\n{scenario}:")
            print(f"  No Exit Preference:   {without_pref_mean:.1f} steps (±{without_pref.iloc[0]['std']:.1f})")
            print(f"  With Exit Preference: {with_pref_mean:.1f} steps (±{with_pref.iloc[0]['std']:.1f})")
            print(f"  Impact: {diff_pct:+.1f}%")



INTERACTION ANALYSIS: SCENARIO TYPE vs EXIT PREFERENCE

CORRIDOR:
------------------------------------------------------------------------------------------------------------------------
  No Exit Preference       : mean=   45.9, std=   10.3, median=   45.0, range=[27, 81]
  With Exit Preference     : mean=  520.8, std=  121.4, median=  516.0, range=[199, 1486]

MALL:
------------------------------------------------------------------------------------------------------------------------
  No Exit Preference       : mean=   48.9, std=   10.0, median=   47.0, range=[29, 96]
  With Exit Preference     : mean=  124.7, std=   25.8, median=  123.0, range=[74, 276]

SEATS:
------------------------------------------------------------------------------------------------------------------------
  No Exit Preference       : mean=  125.0, std=   15.5, median=  124.0, range=[86, 198]
  With Exit Preference     : mean=  130.9, std=   17.7, median=  129.0, range=[94, 202]

SNAKE:
-------------------

### Number of exits

In [40]:
df_n_exits = pd.concat([
    pd.read_csv(experiments['experiments_results_n_exit_impact_mall']),
    pd.read_csv(experiments['experiments_results_n_exit_impact_corridor']),
    pd.read_csv(experiments['experiments_results_n_exit_impact_seats']),
    pd.read_csv(experiments['experiments_results_n_exit_impact_snake']),
])

df_n_exits.groupby(["RunId", "iteration","scenario_type", "n_exits"])["Step"].max()

RunId  iteration  scenario_type  n_exits
0      0          CORRIDOR       1          139
                  MALL           1          118
                  SEATS          1          215
                  SNAKE          1          386
1      0          CORRIDOR       2           84
                                           ... 
7997   999        SEATS          6           98
7998   999        MALL           7           45
                  SEATS          7           95
7999   999        MALL           8           56
                  SEATS          8          135
Name: Step, Length: 22000, dtype: int64

In [41]:

# Get max steps per run by scenario and number of exits
max_steps_n_exits = df_n_exits.groupby(["RunId", "scenario_type", "n_exits"])["Step"].max().reset_index()
max_steps_n_exits.columns = ["RunId", "scenario_type", "n_exits", "max_steps"]
max_steps_n_exits.head(15)


,RunId,scenario_type,n_exits,max_steps
0,0,CORRIDOR,1,139
1,0,MALL,1,118
2,0,SEATS,1,215
3,0,SNAKE,1,386
4,1,CORRIDOR,2,84
5,1,MALL,2,58
6,1,SEATS,2,151
7,1,SNAKE,2,184
8,2,CORRIDOR,3,104
9,2,MALL,3,55


In [42]:

# Overall statistics per number of exits
stats_per_n_exits = max_steps_n_exits.groupby("n_exits")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER NUMBER OF EXITS (All Scenarios Combined)")
print("=" * 80)
print(stats_per_n_exits)

# Statistics by scenario type AND number of exits
print("\n" + "=" * 80)
print("BREAKDOWN BY SCENARIO TYPE AND NUMBER OF EXITS")
print("=" * 80)

scenario_types_n_exits = sorted(df_n_exits['scenario_type'].unique())
for scenario in scenario_types_n_exits:
    scenario_data = max_steps_n_exits[max_steps_n_exits['scenario_type'] == scenario]
    scenario_stats = scenario_data.groupby("n_exits")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario.upper()}:")
    print(scenario_stats)


STATISTICS OF MAX STEPS PER NUMBER OF EXITS (All Scenarios Combined)
         count    mean  median    std  min  max    q25     q75
n_exits                                                       
1         4000  203.47   168.0  94.03   96  519  129.0  250.25
2         4000  122.26   111.0  45.65   52  274   84.0  154.00
3         3000   90.96    85.0  27.82   45  172   68.0  115.00
4         3000   72.85    53.0  38.35   27  187   43.0  114.00
5         2000   80.67    82.0  35.56   28  184   46.0  112.00
6         2000   81.13    84.0  36.66   28  167   45.0  114.00
7         2000   78.01    79.5  34.78   26  180   45.0  108.00
8         2000   79.62    82.0  37.21   23  181   44.0  114.00

BREAKDOWN BY SCENARIO TYPE AND NUMBER OF EXITS

CORRIDOR:
         count    mean  median    std  min  max
n_exits                                        
1         1000  133.18   132.0  12.39  106  178
2         1000   84.72    83.0  14.66   52  149
3         1000   74.40    73.0  16.97   45  142
4 

In [43]:

# Create subplots: one per scenario type with number of exits comparison
num_scenarios_n_exits = len(scenario_types_n_exits)

# Calculate dimensions for subplots (2 columns, rows as needed)
num_cols = 2
num_rows = (num_scenarios_n_exits + num_cols - 1) // num_cols

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[s.upper() for s in scenario_types_n_exits],
    specs=[[{} for _ in range(num_cols)] for _ in range(num_rows)]
)

# Get unique number of exits
n_exits_values = sorted(max_steps_n_exits['n_exits'].unique())
colors_n_exits = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan"]

# Create a bar plot for each scenario
for idx, scenario in enumerate(scenario_types_n_exits):
    row = (idx // num_cols) + 1
    col = (idx % num_cols) + 1
    
    scenario_data = max_steps_n_exits[max_steps_n_exits['scenario_type'] == scenario]
    
    # Add bars for each number of exits
    for exit_idx, n_exit in enumerate(n_exits_values):
        exit_data = scenario_data[scenario_data['n_exits'] == n_exit]
        exit_stats = exit_data.groupby("n_exits")["max_steps"].agg(["mean", "std"]).reset_index()
        
        if len(exit_stats) > 0:
            fig.add_trace(
                go.Bar(
                    x=[f"{int(n_exit)} exits"],
                    y=exit_stats["mean"],
                    error_y=dict(type="data", array=exit_stats["std"], visible=True),
                    name=f"{int(n_exit)} exits",
                    marker=dict(color=colors_n_exits[exit_idx % len(colors_n_exits)]),
                    showlegend=(idx == 0),  # Show legend only for first subplot
                ),
                row=row, col=col
            )
    
    fig.update_xaxes(title_text="Number of Exits", row=row, col=col)
    fig.update_yaxes(title_text="Max Steps", row=row, col=col)

fig.update_layout(height=300*num_rows*3, width=1400, showlegend=True,
                  title_text="Number of Exits Impact Analysis - Max Steps by Scenario")
fig.show()


In [44]:

# Detailed comparison analysis: scenario vs number of exits
print("\n" + "=" * 120)
print("INTERACTION ANALYSIS: SCENARIO TYPE vs NUMBER OF EXITS")
print("=" * 120)

# Calculate statistics for each combination
interaction_stats_n_exits = []
for scenario in scenario_types_n_exits:
    scenario_data = max_steps_n_exits[max_steps_n_exits['scenario_type'] == scenario]
    
    print(f"\n{scenario.upper()}:")
    print("-" * 120)
    
    for n_exit in n_exits_values:
        exit_data = scenario_data[scenario_data['n_exits'] == n_exit]['max_steps']
        
        if len(exit_data) > 0:
            stats_dict = {
                'scenario': scenario,
                'n_exits': n_exit,
                'count': len(exit_data),
                'mean': exit_data.mean(),
                'std': exit_data.std(),
                'median': exit_data.median(),
                'min': exit_data.min(),
                'max': exit_data.max()
            }
            interaction_stats_n_exits.append(stats_dict)
            
            print(f"  {int(n_exit):3.0f} exits: mean={exit_data.mean():7.1f}, std={exit_data.std():7.1f}, "
                  f"median={exit_data.median():7.1f}, range=[{exit_data.min():.0f}, {exit_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df_n_exits = pd.DataFrame(interaction_stats_n_exits)

# Analyze effect of increasing number of exits
print("\n" + "=" * 120)
print("IMPACT ANALYSIS: EFFECT OF INCREASING NUMBER OF EXITS")
print("=" * 120)

for scenario in scenario_types_n_exits:
    print(f"\n{scenario.upper()}:")
    
    scenario_data = interaction_df_n_exits[interaction_df_n_exits['scenario'] == scenario]
    
    if len(scenario_data) > 1:
        baseline = scenario_data[scenario_data['n_exits'] == scenario_data['n_exits'].min()]['mean'].values[0]
        max_mean = scenario_data['mean'].max()
        max_exits = scenario_data[scenario_data['mean'] == max_mean]['n_exits'].values[0]
        min_mean = scenario_data['mean'].min()
        min_exits = scenario_data[scenario_data['mean'] == min_mean]['n_exits'].values[0]
        
        improvement_pct = ((baseline - min_mean) / baseline) * 100
        
        print(f"  Minimum exits ({int(scenario_data['n_exits'].min())}): {baseline:.1f} steps")
        print(f"  Maximum exits ({int(scenario_data['n_exits'].max())}): {min_mean:.1f} steps (best)")
        print(f"  Improvement: {improvement_pct:+.1f}%")

# Number of exits comparison at each scenario
print("\n" + "=" * 120)
print("EVACUATION TIME BY NUMBER OF EXITS - PER SCENARIO")
print("=" * 120)

for scenario in scenario_types_n_exits:
    scenario_data = interaction_df_n_exits[interaction_df_n_exits['scenario'] == scenario]
    
    print(f"\n{scenario}:")
    for _, row in scenario_data.iterrows():
        print(f"  {int(row['n_exits'])} exits: {row['mean']:.1f} steps (±{row['std']:.1f})")



INTERACTION ANALYSIS: SCENARIO TYPE vs NUMBER OF EXITS

CORRIDOR:
------------------------------------------------------------------------------------------------------------------------
    1 exits: mean=  133.2, std=   12.4, median=  132.0, range=[106, 178]
    2 exits: mean=   84.7, std=   14.7, median=   83.0, range=[52, 149]
    3 exits: mean=   74.4, std=   17.0, median=   73.0, range=[45, 142]
    4 exits: mean=   45.6, std=   10.1, median=   45.0, range=[27, 88]

MALL:
------------------------------------------------------------------------------------------------------------------------
    1 exits: mean=  127.0, std=   16.0, median=  124.0, range=[96, 209]
    2 exits: mean=   85.7, std=   14.9, median=   84.5, range=[53, 154]
    3 exits: mean=   74.9, std=   15.4, median=   73.0, range=[45, 126]
    4 exits: mean=   48.5, std=    9.6, median=   47.0, range=[31, 96]
    5 exits: mean=   47.4, std=    9.8, median=   46.0, range=[28, 89]
    6 exits: mean=   46.8, std=   10.1

### Width of exit lines

In [10]:
df_w_exits_0 = pd.read_csv(experiments['experiments_results_open_space_seats_impact_0'])
df_w_exits_0['w_exits'] = 1

df_w_exits_1 = pd.read_csv(experiments['experiments_results_open_space_seats_impact_1'])
df_w_exits_1['w_exits'] = 2

df_w_exits_2 = pd.read_csv(experiments['experiments_results_open_space_seats_impact_2'])
df_w_exits_2['w_exits'] = 3

df_w_exits_3 = pd.read_csv(experiments['experiments_results_open_space_seats_impact_3'])
df_w_exits_3['w_exits'] = 4

df_w_exits = pd.concat([
    df_w_exits_0,
    df_w_exits_1,
    df_w_exits_2,
    df_w_exits_3,
])

In [11]:

df_w_exits.groupby(["RunId", "iteration", "initial_agents", "w_exits"])["Step"].max()
# df_w_exits.head(200)


RunId  iteration  initial_agents  w_exits
0      0          125             1          155
                                  2          122
                                  3           71
                                  4          124
1      1          125             1          160
                                            ... 
998    998        125             4           77
999    999        125             1          116
                                  2          131
                                  3           91
                                  4          104
Name: Step, Length: 4000, dtype: int64

In [12]:

# Get max steps per run by initial agents and exit width
max_steps_w_exits = df_w_exits.groupby(["RunId", "initial_agents", "w_exits"])["Step"].max().reset_index()
max_steps_w_exits.columns = ["RunId", "initial_agents", "w_exits", "max_steps"]
max_steps_w_exits.head(15)


,RunId,initial_agents,w_exits,max_steps
0,0,125,1,155
1,0,125,2,122
2,0,125,3,71
3,0,125,4,124
4,1,125,1,160
5,1,125,2,108
6,1,125,3,83
7,1,125,4,62
8,2,125,1,145
9,2,125,2,106


In [14]:

# Overall statistics per exit width
stats_per_w_exits = max_steps_w_exits.groupby("w_exits")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("=" * 80)
print("STATISTICS OF MAX STEPS PER EXIT WIDTH (All Agent Counts Combined)")
print("=" * 80)
print(stats_per_w_exits)

# Statistics by initial agents AND exit width
print("\n" + "=" * 80)
print("BREAKDOWN BY INITIAL AGENTS AND EXIT WIDTH")
print("=" * 80)

agent_counts_w_exits = sorted(df_w_exits['initial_agents'].unique())
for agent_count in agent_counts_w_exits:
    agent_data = max_steps_w_exits[max_steps_w_exits['initial_agents'] == agent_count]
    agent_stats = agent_data.groupby("w_exits")["max_steps"].agg([
        "count", "mean", "median", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{int(agent_count)} agents:")
    print(agent_stats)


STATISTICS OF MAX STEPS PER EXIT WIDTH (All Agent Counts Combined)
         count    mean  median    std  min  max    q25    q75
w_exits                                                      
1         1000  142.31   142.0  17.26   99  209  130.0  153.0
2         1000  103.00   102.0  18.97   65  185   90.0  115.0
3         1000  101.19   100.0  19.44   59  160   87.0  113.0
4         1000   99.81    98.0  19.57   53  166   86.0  112.0

BREAKDOWN BY INITIAL AGENTS AND EXIT WIDTH

125 agents:
         count    mean  median    std  min  max
w_exits                                        
1         1000  142.31   142.0  17.26   99  209
2         1000  103.00   102.0  18.97   65  185
3         1000  101.19   100.0  19.44   59  160
4         1000   99.81    98.0  19.57   53  166


In [15]:

# Create subplots: one per agent count with exit width comparison
num_agent_counts_w_exits = len(agent_counts_w_exits)

# Calculate dimensions for subplots (2 columns, rows as needed)
num_cols = 2
num_rows = (num_agent_counts_w_exits + num_cols - 1) // num_cols

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[f"{int(ac)} agents" for ac in agent_counts_w_exits],
    specs=[[{} for _ in range(num_cols)] for _ in range(num_rows)]
)

# Get unique exit widths
w_exits_values = sorted(max_steps_w_exits['w_exits'].unique())
colors_w_exits = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan"]

# Create a line plot for each agent count
for idx, agent_count in enumerate(agent_counts_w_exits):
    row = (idx // num_cols) + 1
    col = (idx % num_cols) + 1
    
    agent_data = max_steps_w_exits[max_steps_w_exits['initial_agents'] == agent_count]
    agent_stats = agent_data.groupby("w_exits")["max_steps"].agg(["mean", "std"]).reset_index()
    
    fig.add_trace(
        go.Scatter(
            x=agent_stats["w_exits"],
            y=agent_stats["mean"],
            error_y=dict(type="data", array=agent_stats["std"], visible=True),
            mode="lines+markers",
            name=f"{int(agent_count)} agents",
            marker=dict(size=8),
            line=dict(width=2, color=colors_w_exits[idx % len(colors_w_exits)]),
            showlegend=True
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(title_text="Exit Width Multiplier", row=row, col=col)
    fig.update_yaxes(title_text="Max Steps", row=row, col=col)

fig.update_layout(height=300*num_rows, width=1400, showlegend=True,
                  title_text="Exit Width Impact Analysis - Max Steps by Agent Count")
fig.show()


In [16]:

# Detailed comparison analysis: agent count vs exit width
print("\n" + "=" * 120)
print("INTERACTION ANALYSIS: INITIAL AGENTS vs EXIT WIDTH")
print("=" * 120)

# Calculate statistics for each combination
interaction_stats_w_exits = []
for agent_count in agent_counts_w_exits:
    agent_data = max_steps_w_exits[max_steps_w_exits['initial_agents'] == agent_count]
    
    print(f"\n{int(agent_count)} agents:")
    print("-" * 120)
    
    for w_exit in w_exits_values:
        exit_data = agent_data[agent_data['w_exits'] == w_exit]['max_steps']
        
        if len(exit_data) > 0:
            stats_dict = {
                'initial_agents': agent_count,
                'w_exits': w_exit,
                'count': len(exit_data),
                'mean': exit_data.mean(),
                'std': exit_data.std(),
                'median': exit_data.median(),
                'min': exit_data.min(),
                'max': exit_data.max()
            }
            interaction_stats_w_exits.append(stats_dict)
            
            print(f"  Width {int(w_exit):3.0f}: mean={exit_data.mean():7.1f}, std={exit_data.std():7.1f}, "
                  f"median={exit_data.median():7.1f}, range=[{exit_data.min():.0f}, {exit_data.max():.0f}]")

# Convert to DataFrame for analysis
interaction_df_w_exits = pd.DataFrame(interaction_stats_w_exits)

# Analyze effect of increasing exit width at each agent count
print("\n" + "=" * 120)
print("IMPACT ANALYSIS: EFFECT OF INCREASING EXIT WIDTH")
print("=" * 120)

for agent_count in agent_counts_w_exits:
    print(f"\n{int(agent_count)} agents:")
    
    agent_data = interaction_df_w_exits[interaction_df_w_exits['initial_agents'] == agent_count]
    
    if len(agent_data) > 1:
        baseline = agent_data[agent_data['w_exits'] == agent_data['w_exits'].min()]['mean'].values[0]
        max_mean = agent_data['mean'].max()
        max_width = agent_data[agent_data['mean'] == max_mean]['w_exits'].values[0]
        min_mean = agent_data['mean'].min()
        min_width = agent_data[agent_data['mean'] == min_mean]['w_exits'].values[0]
        
        improvement_pct = ((baseline - min_mean) / baseline) * 100
        
        print(f"  Minimum width ({int(agent_data['w_exits'].min())}): {baseline:.1f} steps")
        print(f"  Maximum width ({int(agent_data['w_exits'].max())}): {min_mean:.1f} steps (best)")
        print(f"  Improvement: {improvement_pct:+.1f}%")

# Exit width comparison at each agent count
print("\n" + "=" * 120)
print("EVACUATION TIME BY EXIT WIDTH - PER AGENT COUNT")
print("=" * 120)

for agent_count in agent_counts_w_exits:
    agent_data = interaction_df_w_exits[interaction_df_w_exits['initial_agents'] == agent_count]
    
    print(f"\n{int(agent_count)} agents:")
    for _, row in agent_data.iterrows():
        print(f"  Width {int(row['w_exits'])}: {row['mean']:.1f} steps (±{row['std']:.1f})")



INTERACTION ANALYSIS: INITIAL AGENTS vs EXIT WIDTH

125 agents:
------------------------------------------------------------------------------------------------------------------------
  Width   1: mean=  142.3, std=   17.3, median=  142.0, range=[99, 209]
  Width   2: mean=  103.0, std=   19.0, median=  102.0, range=[65, 185]
  Width   3: mean=  101.2, std=   19.4, median=  100.0, range=[59, 160]
  Width   4: mean=   99.8, std=   19.6, median=   98.0, range=[53, 166]

IMPACT ANALYSIS: EFFECT OF INCREASING EXIT WIDTH

125 agents:
  Minimum width (1): 142.3 steps
  Maximum width (4): 99.8 steps (best)
  Improvement: +29.9%

EVACUATION TIME BY EXIT WIDTH - PER AGENT COUNT

125 agents:
  Width 1: 142.3 steps (±17.3)
  Width 2: 103.0 steps (±19.0)
  Width 3: 101.2 steps (±19.4)
  Width 4: 99.8 steps (±19.6)
